##### GOLD датасет состоять будет из:
- примерно 800 предложений, которые разметил Разметчик
    - вычислить их легко - там где будет `spans_rus` или `relations_rus` непустой
- предложений, которые были отправлены на простую проверку перевода
    - предварительно собрать ID предложений, отправленных на проверку перевода
    - собрать данные по этим ID

Предполагается, что оригинальный датасет лежит в директории `../hf_deft_ru/`

In [1]:
import json
from tqdm import tqdm
import logging
from datasets import Dataset
import datetime


/home/dpopov/work/rudeft/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Настройка логгера, для вывода подробной информации работа скрипта в файл `log.log`

При этом основной вывод прогрессбара от `tqdm` будет осуществляться в `Out` ячейку

In [2]:
# Настройка логгера для записи в файл
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)

# Создание обработчика для записи в файл
file_handler = logging.FileHandler('log.log')
file_handler.setLevel(logging.DEBUG)

# Создание форматтера для сообщений
FORMAT = "%(asctime)s %(levelname)s [%(process)s]: %(name)s: %(message)s"
formatter = logging.Formatter(FORMAT)
file_handler.setFormatter(formatter)

# Добавление обработчика к логгеру
logger.addHandler(file_handler)

In [3]:
# orig files
dev_file = "../hf_deft_ru/dev.json"
train_file = "../hf_deft_ru/train.json"

In [4]:
ids_verified_sent_file = "../convert_translation_ls_to_hf/ids_verified_sent_07_10_2024_17_22.json"

In [5]:
def make_gold_dataset(hf_file_dev, hf_file_train, gold_file, ids_verified_sent_file):
    """ Функция создания GOLD датасета для перевода в HF (Hugging Face) формате.

    
    :param hf_file_dev: путь к dev файлу датасета в HF формате.
    :param hf_file_train: путь к train файлу датасета в HF формате.
    :param gold_file: путь к файлу выгрузки размеченных данных в LS формате.
    :param ids_verified_sent_file: путь к файлу со списком ID верифицированных переводов.

    """
    cnt_processed = 0
    cnt_added = 0
    cnt_added_in_verified_sent = 0
    cnt_added_by_annotator = 0
    cnt_skip = 0



    # считываем данные из hf датасета   
    hf_data_dev = Dataset.from_json(hf_file_dev)        # list of dicts (dict = one line from hf dataset)
    hf_data_dev = hf_data_dev.to_list()
    hf_data_train = Dataset.from_json(hf_file_train)        # list of dicts (dict = one line from hf dataset)
    hf_data_train = hf_data_train.to_list()
    hf_data = hf_data_dev.copy()
    hf_data.extend(hf_data_train)                   # all data

    
    # создаем индекс для быстрого доступа к нужному элементу HF по ID
    id2hfindex = {}
    for index, elem in tqdm(enumerate(hf_data), total=len(hf_data), desc="Create index id2hfindex"):
        id2hfindex[elem["id"]] = index

    # считываем id верифицированных переводов
    with open(ids_verified_sent_file, 'r') as f:
        ids_verified_sent = json.load(f)
    ids_verified_sent = set(ids_verified_sent)
    
    gold_data = []
    for hf_elem in tqdm(hf_data, desc="Processing HF data"):
        cnt_processed += 1

        hf_id = hf_elem['id']

        if hf_id in ids_verified_sent:          # если документ среди верифицированных, то записываем в GOLD
            gold_data.append(hf_elem)
            logging.info(f"HF. ID: '{hf_id}' is in ids_verified_sent! Add to GOLD!")
            cnt_added_in_verified_sent += 1
            continue

        if ( 'spans_rus' in hf_elem and hf_elem['spans_rus'] ) or ( 'relations_rus' in hf_elem and hf_elem['relations_rus'] ):
            # Размеченные Разметчиком - записываем в GOLD
            gold_data.append(hf_elem)
            logging.info(f"HF. ID: '{hf_id}' verified by annotator! Add to GOLD!")
            cnt_added_by_annotator += 1
            continue
        
        cnt_skip += 1

    cnt_added = cnt_added_in_verified_sent + cnt_added_by_annotator

    logging.info(f"Processed: {cnt_processed}, \tAdded: {cnt_added}, \tAdded_in_verified_sent: {cnt_added_in_verified_sent}, \tAdded_by_annotator: {cnt_added_by_annotator}, \tSkip_total: {cnt_skip}, \tLen_gold_data: {len(gold_data)}.")
    
    # Записываем данные в HF датасет
    dataset = Dataset.from_list(gold_data)
    dataset.to_json(gold_file)



In [7]:
gold_file = "./gold_for_translation.json"

In [8]:
make_gold_dataset(dev_file, train_file, gold_file, ids_verified_sent_file)

Creating json from Arrow format: 100%|██████████| 2/2 [00:00<00:00, 24.37ba/s]


In [9]:
!grep "Processed:" log.log 

2024-10-07 20:25:56,930 INFO [616304]: root: Processed: 25363, 	Added: 1070, 	Added_in_verified_sent: 200, 	Added_by_annotator: 870, 	Skip_total: 24293, 	Len_gold_data: 1070.
